In [1]:
# Enable autoreload - automatically reimport modules when they change
%load_ext autoreload
%autoreload 2

print("✅ Autoreload enabled - changes to src/arcflow/ will be picked up automatically!")

✅ Autoreload enabled - changes to src/arcflow/ will be picked up automatically!


In [10]:
spark = SparkSession.getActiveSession()

In [11]:
spark.range(1).collect()

[Row(id=0)]

In [ ]:
import os
from pyspark.sql import SparkSession

# Stop existing session if any
try:
    spark.stop()  # noqa: F821
    print("🛑 Stopped existing Spark session")
except:
    pass

# Configure Delta Lake with builder pattern
# The key is to use configure_spark_with_delta_pip() from delta.pip_utils
from delta import configure_spark_with_delta_pip
warehouse_dir = os.path.abspath(r'C:\Users\milescole\source\ArcFlow\dev\storage\lakehouse')
builder = (SparkSession.builder
    .appName("ArcFlow Example")
    .config("spark.driver.memory", "4g")
    .master("local[*]")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.warehouse.dir", warehouse_dir)
    .config("spark.driver.host", "localhost")
    .config("spark.driver.bindAddress", "localhost")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.catalogImplementation", "hive")
    # for windows
    .config("spark.hadoop.io.native.lib.available", "false")
    .config("spark.hadoop.fs.file.impl.disable.cache", "true")
)

# This automatically configures all Delta settings and downloads JARs
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f"✅ Spark session created with Delta Lake support")
print(f"   Spark version: {spark.version}")
print(f"   App name: {spark.sparkContext.appName}")
print(f"   Master: {spark.sparkContext.master}")

✅ Spark session created with Delta Lake support
   Spark version: 3.5.7
   App name: ArcFlow Example
   Master: local[*]


In [ ]:
from pyspark.sql import DataFrame
import pyspark.sql.functions as sf
from arcflow import SourceConfig, ZoneConfig
from arcflow.transformations.zone_transforms import register_zone_transformer
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType,
    ArrayType,
    TimestampType,
    BooleanType,
)


@register_zone_transformer
def explode_data(df: DataFrame) -> DataFrame:
    return df.selectExpr("_meta", "explode(data) as data")


# Configure tables
tables = {}


@register_zone_transformer
def silver_item(df) -> DataFrame:
    df = (
        df.withColumn(
            "is_sdofcertified", sf.col("is_sdofcertified").cast(BooleanType())
        )
        .withColumn("generated_at", sf.col("generated_at").cast(TimestampType()))
        .drop("_processing_timestamp")
    )

    return df


tables["item"] = SourceConfig(
    name="item",
    format="parquet",
    source_uri=r"C:\Users\milescole\source\LakeGen\dev\output_test\mcmillian\item",
    schema=StructType(
        [
            StructField("ItemId", StringType(), True),
            StructField("SKU", StringType(), True),
            StructField("Description", StringType(), True),
            StructField("Brand", StringType(), True),
            StructField("Category", StringType(), True),
            StructField("Subcategory", StringType(), True),
            StructField("Material", StringType(), True),
            StructField("NominalSize", DoubleType(), True),
            StructField("EndConnection", StringType(), True),
            StructField("PressureClass", LongType(), True),
            StructField("Weight", DoubleType(), True),
            StructField("Cost", DoubleType(), True),
            StructField("ListPrice", DoubleType(), True),
            StructField("IsSDOFCertified", StringType(), True),
            StructField("StructuralIndex", DoubleType(), True),
            StructField("SpanRating", DoubleType(), True),
            StructField("OrganizationId", LongType(), True),
            StructField("GeneratedAt", StringType(), True),
        ]
    ),
    description="Item master data",
    trigger_mode='processingTime',
    zones={
        "bronze": ZoneConfig(
            enabled=True,
            mode="append",
        ),
        "silver": ZoneConfig(
            enabled=True,
            mode="upsert",
            merge_keys=["item_id"],
            custom_transform="silver_item",
        )
    }
)


@register_zone_transformer
def silver_shipment(df) -> DataFrame:
    df = (
        df.selectExpr(
            "_meta.*",
            "data.*",
        )
        .withColumn(
            "committed_delivery_date", sf.to_timestamp("committed_delivery_date")
        )
        .withColumn("ship_date", sf.to_timestamp("ship_date"))
        .withColumn("generated_at", sf.to_timestamp("generated_at"))
        .withColumn("enqueued_time", sf.to_timestamp("enqueued_time"))
        .drop("_meta", "data", "_processing_timestamp")
    )

    return df


tables["shipment"] = SourceConfig(
    name="shipment",
    format="json",
    source_uri=r"C:\Users\milescole\source\LakeGen\dev\output_test\mcmillian\shipment",
    schema=StructType(
        [
            StructField(
                "_meta",
                StructType(
                    [
                        StructField("enqueuedTime", StringType(), True),
                        StructField("producer", StringType(), True),
                        StructField("recordType", StringType(), True),
                        StructField("schemaVersion", StringType(), True),
                    ]
                ),
                True,
            ),
            StructField(
                "data",
                ArrayType(
                    StructType(
                        [
                            StructField("CommittedDeliveryDate", StringType(), True),
                            StructField("CurrentFacilityId", StringType(), True),
                            StructField("CustomerId", StringType(), True),
                            StructField("CustomerName", StringType(), True),
                            StructField("DeclaredValue", DoubleType(), True),
                            StructField("DestinationAddress", StringType(), True),
                            StructField("DestinationCity", StringType(), True),
                            StructField("DestinationCountry", StringType(), True),
                            StructField("DestinationFacilityId", StringType(), True),
                            StructField("DestinationLatitude", DoubleType(), True),
                            StructField("DestinationLongitude", DoubleType(), True),
                            StructField("DestinationState", StringType(), True),
                            StructField("DestinationZipCode", StringType(), True),
                            StructField("Distance", DoubleType(), True),
                            StructField("GeneratedAt", StringType(), True),
                            StructField("Height", DoubleType(), True),
                            StructField("IsFragile", BooleanType(), True),
                            StructField("IsHazardous", BooleanType(), True),
                            StructField(
                                "LateDeliveryPenaltyPerDay", DoubleType(), True
                            ),
                            StructField("Length", DoubleType(), True),
                            StructField("OrderId", StringType(), True),
                            StructField(
                                "OrderLineList", ArrayType(LongType(), True), True
                            ),
                            StructField("OrganizationId", LongType(), True),
                            StructField("OriginAddress", StringType(), True),
                            StructField("OriginCity", StringType(), True),
                            StructField("OriginCountry", StringType(), True),
                            StructField("OriginFacilityId", StringType(), True),
                            StructField("OriginLatitude", DoubleType(), True),
                            StructField("OriginLongitude", DoubleType(), True),
                            StructField("OriginState", StringType(), True),
                            StructField("OriginZipCode", StringType(), True),
                            StructField("RequiresRefrigeration", BooleanType(), True),
                            StructField("ServiceLevel", StringType(), True),
                            StructField("ShipDate", StringType(), True),
                            StructField("ShipmentId", StringType(), True),
                            StructField("SpecialInstructions", StringType(), True),
                            StructField("TrackingNumber", StringType(), True),
                            StructField("Volume", DoubleType(), True),
                            StructField("Weight", DoubleType(), True),
                            StructField("Width", DoubleType(), True),
                        ]
                    ),
                    True,
                ),
                True,
            ),
        ]
    ),
    description="Shipment header",
    trigger_mode='processingTime',
    zones={
        "bronze": ZoneConfig(
            enabled=True, mode="append", custom_transform="explode_data"
        ),
        "silver": ZoneConfig(
            enabled=True,
            mode="upsert",
            merge_keys=["shipment_id"],
            custom_transform="silver_shipment",
        )
    }
)


@register_zone_transformer
def silver_shipment_scan_event(df) -> DataFrame:
    df = (
        df.selectExpr(
            "_meta.*",
            "data.*",
        )
        .withColumn("enqueued_time", sf.to_timestamp("enqueued_time"))
        .withColumn("generated_at", sf.to_timestamp("generated_at"))
        .withColumn("event_timestamp", sf.to_timestamp("event_timestamp"))
        .withColumn("delivery_review", sf.col("additional_data").getItem("review"))
        .drop("_meta", "data", "_processing_timestamp")
    )

    return df


tables["shipment_scan_event"] = SourceConfig(
    name="shipment_scan_event",
    format="json",
    source_uri=r"C:\Users\milescole\source\LakeGen\dev\output_test\mcmillian\shipment_scan_event",
    schema=StructType(
        [
            StructField(
                "_meta",
                StructType(
                    [
                        StructField("enqueuedTime", StringType(), True),
                        StructField("producer", StringType(), True),
                        StructField("recordType", StringType(), True),
                        StructField("schemaVersion", StringType(), True),
                    ]
                ),
                True,
            ),
            StructField(
                "data",
                ArrayType(
                    StructType(
                        [
                            StructField(
                                "AdditionalData",
                                StructType(
                                    [
                                        StructField("condition", StringType(), True),
                                        StructField("loadId", StringType(), True),
                                        StructField("method", StringType(), True),
                                        StructField("note", StringType(), True),
                                        StructField("reason", StringType(), True),
                                        StructField("reroute", StringType(), True),
                                        StructField("resolution", StringType(), True),
                                        StructField("review", StringType(), True),
                                        StructField("signedBy", StringType(), True),
                                        StructField("sortDecision", StringType(), True),
                                        StructField("stopSequence", LongType(), True),
                                        StructField(
                                            "transportType", StringType(), True
                                        ),
                                        StructField("vehicleId", StringType(), True),
                                    ]
                                ),
                                True,
                            ),
                            StructField(
                                "CurrentDestinationFacilityId", StringType(), True
                            ),
                            StructField("CurrentOriginFacilityId", StringType(), True),
                            StructField("CurrentServiceLevel", StringType(), True),
                            StructField("EmployeeId", StringType(), True),
                            StructField("EstimatedArrivalTime", StringType(), True),
                            StructField("EventId", StringType(), True),
                            StructField("EventTimestamp", StringType(), True),
                            StructField("EventType", StringType(), True),
                            StructField("ExceptionCode", StringType(), True),
                            StructField("ExceptionSeverity", StringType(), True),
                            StructField("FacilityId", StringType(), True),
                            StructField("GeneratedAt", StringType(), True),
                            StructField("LocationLatitude", DoubleType(), True),
                            StructField("LocationLongitude", DoubleType(), True),
                            StructField("NextWaypointFacilityId", StringType(), True),
                            StructField("OrganizationId", LongType(), True),
                            StructField(
                                "PlannedPathSnapshot",
                                ArrayType(StringType(), True),
                                True,
                            ),
                            StructField("RelatedExceptionEventId", StringType(), True),
                            StructField("ResolutionAction", StringType(), True),
                            StructField("RouteId", StringType(), True),
                            StructField("ScanDeviceId", StringType(), True),
                            StructField("ScheduleId", StringType(), True),
                            StructField("SequenceNumber", LongType(), True),
                            StructField("ShipmentId", StringType(), True),
                            StructField("SortLaneId", StringType(), True),
                            StructField("SortingEquipmentId", StringType(), True),
                            StructField("TrackingNumber", StringType(), True),
                        ]
                    ),
                    True,
                ),
                True,
            ),
        ]
    ),
    description="Events tracking shipment to destination",
    trigger_mode='processingTime',
    zones={
        "bronze": ZoneConfig(
            enabled=True, mode="append", custom_transform="explode_data"
        ),
        "silver": ZoneConfig(
            enabled=True,
            mode="upsert",
            merge_keys=["event_id"],
            custom_transform="silver_shipment_scan_event",
        )
    }
)

@register_zone_transformer
def silver_order_transformer(df) -> DataFrame:
    df = (
        df.withColumn("order_lines", sf.explode_outer("order_lines"))
            .select("*", "order_lines.*")
            .withColumn("order_date", sf.to_timestamp("order_date", "yyyy-MM-dd"))
            .withColumn("generated_at", sf.to_timestamp("generated_at"))
            .drop("_processing_timestamp", "order_lines")
    )

    return df


tables["order"] = SourceConfig(
    name="order",
    format="parquet",
    source_uri=r"C:\Users\milescole\source\LakeGen\dev\output_test\mcmillian\order",
    schema=StructType([StructField('OrderId', StringType(), True), StructField('OrderNumber', StringType(), True), StructField('OrderDate', StringType(), True), StructField('OrderTotal', DoubleType(), True), StructField('Source', StringType(), True), StructField('CustomerId', StringType(), True), StructField('OrderLines', ArrayType(StructType([StructField('Description', StringType(), True), StructField('ExtendedPrice', DoubleType(), True), StructField('ItemId', StringType(), True), StructField('LineNumber', LongType(), True), StructField('NetWeight', DoubleType(), True), StructField('Quantity', LongType(), True), StructField('SKU', StringType(), True), StructField('UnitPrice', DoubleType(), True), StructField('WarrantyIncluded', BooleanType(), True)]), True), True), StructField('OrganizationId', LongType(), True), StructField('GeneratedAt', StringType(), True)]),
    description="Orders and order lines",
    trigger_mode='processingTime',
    zones={
        "bronze": ZoneConfig(
            enabled=True, mode="append"
        ),
        "silver": ZoneConfig(
            enabled=True,
            mode="upsert",
            merge_keys=["order_id", "line_number"],
            custom_transform="silver_order_transformer",
        )
    }
)

@register_zone_transformer
def silver_customer_transformer(df) -> DataFrame:
    df = (
        df.select(
                "*",
                sf.col("delivery_address.address").alias("delivery_address"),
                sf.col("delivery_address.city").alias("delivery_city"),
                sf.col("delivery_address.country").alias("delivery_country"),
                sf.col("delivery_address.latitude").alias("delivery_latitude"),
                sf.col("delivery_address.longitude").alias("delivery_longitude"),
                sf.col("delivery_address.state").alias("delivery_state"),
                sf.col("delivery_address.zip_code").alias("delivery_zip_code"),

                sf.col("billing_address.address").alias("billing_address"),
                sf.col("billing_address.city").alias("billing_city"),
                sf.col("billing_address.country").alias("billing_country"),
                sf.col("billing_address.state").alias("billing_state"),
            )
            .drop("delivery_address", "billing_address")
            .withColumn("generated_at", sf.col("generated_at").cast(TimestampType()))
    )

    return df


tables["customer"] = SourceConfig(
    name="customer",
    format="parquet",
    source_uri=r"C:\Users\milescole\source\LakeGen\dev\output_test\mcmillian\customer",
    schema=StructType([StructField('CustomerId', StringType(), True), StructField('CustomerName', StringType(), True), StructField('Description', StringType(), True), StructField('PrimaryContactFirstName', StringType(), True), StructField('PrimaryContactLastName', StringType(), True), StructField('PrimaryContactEmail', StringType(), True), StructField('PrimaryContactPhone', StringType(), True), StructField('DeliveryAddress', StructType([StructField('Address', StringType(), True), StructField('City', StringType(), True), StructField('Country', StringType(), True), StructField('Latitude', DoubleType(), True), StructField('Longitude', DoubleType(), True), StructField('State', StringType(), True), StructField('ZipCode', StringType(), True)]), True), StructField('BillingAddress', StructType([StructField('Address', StringType(), True), StructField('City', StringType(), True), StructField('Country', StringType(), True), StructField('State', StringType(), True), StructField('ZipCode', StringType(), True)]), True), StructField('OrganizationId', LongType(), True), StructField('GeneratedAt', StringType(), True)]),
    description="Customer master",
    trigger_mode='processingTime',
    zones={
        "bronze": ZoneConfig(
            enabled=True, mode="append"
        ),
        "silver": ZoneConfig(
            enabled=True,
            mode="upsert",
            merge_keys=["customer_id"],
            custom_transform="silver_customer_transformer",
        )
    }
)

@register_zone_transformer
def cast_generated_at(df) -> DataFrame:
    return df.withColumn("generated_at", sf.col("generated_at").cast(TimestampType()))

tables["service_level"] = SourceConfig(
    name="service_level",
    format="parquet",
    source_uri=r"C:\Users\milescole\source\LakeGen\dev\output_test\mcmillian\servicelevel",
    schema=StructType([StructField('level', StringType(), True), StructField('weight', LongType(), True), StructField('days', LongType(), True), StructField('OrganizationId', LongType(), True), StructField('GeneratedAt', StringType(), True)]),
    description="Shipment service level",
    trigger_mode='availableNow',
    zones={
        "bronze": ZoneConfig(
            enabled=True, mode="append"
        ),
        "silver": ZoneConfig(
            enabled=True,
            mode="upsert",
            merge_keys=["level"],
            custom_transform="cast_generated_at"
        )
    }
)

tables["exception_type"] = SourceConfig(
    name="exception_type",
    format="parquet",
    source_uri=r"C:\Users\milescole\source\LakeGen\dev\output_test\mcmillian\exceptiontype",
    schema=StructType([StructField('code', StringType(), True), StructField('severity', StringType(), True), StructField('description', StringType(), True), StructField('OrganizationId', LongType(), True), StructField('GeneratedAt', StringType(), True)]),
    description="Shipment exception type",
    trigger_mode='availableNow',
    zones={
        "bronze": ZoneConfig(
            enabled=True, mode="append"
        ),
        "silver": ZoneConfig(
            enabled=True,
            mode="upsert",
            merge_keys=["code"],
            custom_transform="cast_generated_at"
        )
    }
)

tables["facility"] = SourceConfig(
    name="facility",
    format="parquet",
    source_uri=r"C:\Users\milescole\source\LakeGen\dev\output_test\mcmillian\facility",
    schema=StructType([StructField('FacilityId', StringType(), True), StructField('FacilityName', StringType(), True), StructField('FacilityType', StringType(), True), StructField('Address', StringType(), True), StructField('City', StringType(), True), StructField('State', StringType(), True), StructField('ZipCode', StringType(), True), StructField('Country', StringType(), True), StructField('Latitude', DoubleType(), True), StructField('Longitude', DoubleType(), True), StructField('OrganizationId', LongType(), True), StructField('GeneratedAt', StringType(), True)]),
    description="Shipment facility",
    trigger_mode='availableNow',
    zones={
        "bronze": ZoneConfig(
            enabled=True, mode="append"
        ),
        "silver": ZoneConfig(
            enabled=True,
            mode="upsert",
            merge_keys=["facility_id"],
            custom_transform="cast_generated_at"
        )
    }
)

tables["route"] = SourceConfig(
    name="route",
    format="parquet",
    source_uri=r"C:\Users\milescole\source\LakeGen\dev\output_test\mcmillian\route",
    schema=StructType([StructField('RouteId', StringType(), True), StructField('OriginFacilityId', StringType(), True), StructField('DestinationFacilityId', StringType(), True), StructField('TransportMode', StringType(), True), StructField('TravelTimeHours', DoubleType(), True), StructField('OrganizationId', LongType(), True), StructField('GeneratedAt', StringType(), True)]),
    description="Shipment routes",
    trigger_mode='availableNow',
    zones={
        "bronze": ZoneConfig(
            enabled=True, mode="append"
        ),
        "silver": ZoneConfig(
            enabled=True,
            mode="upsert",
            merge_keys=["route_id"],
            custom_transform="cast_generated_at"
        )
    }
)

In [ ]:
# Option 1: Import from the parent directory as an absolute import
import sys
from pathlib import Path

# Add parent directory to path
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

# Now import
from pipeline_config import tables

# Option 2: Or just define tables here in the notebook (recommended for notebooks)
# See the tables definition in the previous cell

Overwriting existing transformer: explode_data
Overwriting existing transformer: silver_item
Overwriting existing transformer: silver_shipment
Overwriting existing transformer: silver_shipment_scan_event
Overwriting existing transformer: silver_order_transformer
Overwriting existing transformer: silver_customer_transformer
Overwriting existing transformer: cast_generated_at
Overwriting existing transformer: silver_item
Overwriting existing transformer: silver_shipment
Overwriting existing transformer: silver_shipment_scan_event
Overwriting existing transformer: silver_order_transformer
Overwriting existing transformer: silver_customer_transformer
Overwriting existing transformer: cast_generated_at


In [ ]:
from arcflow import Controller
#from arcflow.core.spark_session import create_spark_session

# Create Spark session
#spark = create_spark_session(app_name="MyELT")

# Configure pipeline
config = {
    'streaming_enabled': True,
    'checkpoint_uri': r'C:\Users\milescole\source\ArcFlow\dev\storage\checkpoints',
    'landing_uri': r"C:\Users\milescole\source\LakeGen\dev\output_test\mcmillian",
    'trigger_interval': '1 seconds' # default if not set at table level
}

# Initialize controller
controller = Controller(
    spark=spark,
    config=config,
    table_registry=tables
)

# Run full pipeline
controller.run_full_pipeline(zones=['bronze', 'silver'])

In [8]:
controller.stop_all()

## Check Stream Status

After running the pipeline, you can check the status of all active streams:

In [9]:
controller.get_status()

{'bronze_item_stream': {'active': False,
  'id': '6e3c2a25-032d-4be3-a1b3-88a234079850',
  'runId': 'f8f36c30-931c-494a-bdc2-a459c24926da',
  'recent_progress': {'id': '6e3c2a25-032d-4be3-a1b3-88a234079850',
   'runId': 'f8f36c30-931c-494a-bdc2-a459c24926da',
   'name': 'bronze_item_stream',
   'timestamp': '2025-12-12T23:23:32.000Z',
   'batchId': 2,
   'numInputRows': 0,
   'inputRowsPerSecond': 0.0,
   'processedRowsPerSecond': 0.0,
   'durationMs': {'latestOffset': 2, 'triggerExecution': 2},
   'stateOperators': [],
   'sources': [{'description': 'FileStreamSource[file:/C:/Users/milescole/source/LakeGen/dev/output_test/mcmillian/item]',
     'startOffset': {'logOffset': 1},
     'endOffset': {'logOffset': 1},
     'latestOffset': None,
     'numInputRows': 0,
     'inputRowsPerSecond': 0.0,
     'processedRowsPerSecond': 0.0}],
   'sink': {'description': 'DeltaSink[file:/C:/Users/milescole/source/demo-industrial-streaming/dev/storage/lakehouse/bronze.db/item]',
    'numOutputRows':

In [5]:
# Check status of all streams
status = controller.get_status()

print(f"Total streams: {status['total']}")
print(f"Active streams: {status['active']}")
print(f"Completed streams: {status['completed']}")
print(f"Failed streams: {status['failed']}")

# Show details for each stream
print("\n📊 Stream Details:")
for query_info in status['queries']:
    print(f"\n  {query_info['name']}")
    print(f"    Status: {query_info['status']}")
    print(f"    Is Active: {query_info['is_active']}")
    if query_info.get('recent_progress'):
        progress = query_info['recent_progress']
        print(f"    Processed: {progress.get('numInputRows', 0)} rows")
        print(f"    Batch: {progress.get('batchId', 'N/A')}")

KeyError: 'total'

In [ ]:
# Alternative: Check individual query status directly
# (useful if you only ran a single table)

from arcflow.pipelines.zone_pipeline import ZonePipeline

pipeline = ZonePipeline(spark, zone="bronze", config=config)
query = pipeline.process_table(tables["shipment"])

if query:
    print(f"Query Name: {query.name}")
    print(f"Query ID: {query.id}")
    print(f"Is Active: {query.isActive}")
    print(f"Status: {query.status}")
    
    # Get recent progress
    progress = query.recentProgress
    if progress:
        latest = progress[-1]
        print(f"\nLatest Batch:")
        print(f"  Batch ID: {latest.get('batchId')}")
        print(f"  Input Rows: {latest.get('numInputRows')}")
        print(f"  Processing Time: {latest.get('durationMs', {}).get('triggerExecution')} ms")

In [ ]:
# Wait for all availableNow triggers to complete
# (This is safe for notebooks - only blocks until availableNow streams finish)
controller.await_completion()

print("✅ All streams completed!")

## Quick Preview: Pre-Write Output

Use `ZonePipeline._apply_transformations()` to see exactly what will be written

In [8]:
from arcflow.pipelines.zone_pipeline import ZonePipeline

# Simple test using the built-in test_output() method!
# It automatically:
# - Calculates source_zone from table config
# - Reads in batch mode (even if pipeline configured for streaming)
# - Applies all transformations (standard + custom)

pipeline = ZonePipeline(
    spark=spark,
    zone="silver",
    config={'streaming_enabled': True}
)

# Test the output - returns a batch DataFrame
df_preview = pipeline.test_output(tables["item"])

df_preview.show()

+--------------------+--------------------+--------------------+--------------------+----------------+-----------------+----------------+------------+-----------------+--------------+------+------+----------+----------------+----------------+-----------+---------------+--------------------+---------------------+
|             item_id|                 sku|         description|               brand|        category|      subcategory|        material|nominal_size|   end_connection|pressure_class|weight|  cost|list_price|is_sdofcertified|structural_index|span_rating|organization_id|        generated_at|_processing_timestamp|
+--------------------+--------------------+--------------------+--------------------+----------------+-----------------+----------------+------------+-----------------+--------------+------+------+----------+----------------+----------------+-----------+---------------+--------------------+---------------------+
|40a112d1-90dc-461...|DY-2.5-FLO-227182155|2.5" Flow Regul

## Process Single Table with availableNow Trigger

The `availableNow` trigger processes all currently available data and then stops - perfect for testing streaming pipelines!

In [ ]:
from arcflow.pipelines.zone_pipeline import ZonePipeline

# Create a streaming pipeline for bronze zone
pipeline = ZonePipeline(
    spark=spark,
    zone="bronze",
    config={
        'streaming_enabled': True,  # Enable streaming mode
        'checkpoint_uri': r'C:\Users\milescole\source\ArcFlow\dev\storage\checkpoints'
    }
)

query = pipeline.process_table(tables["shipment"])


if query:
    print(f"✅ Started streaming query: {query.name}")
    print(f"   Status: {query.status}")
    
    # Wait for the query to finish (availableNow will complete automatically)
    query.awaitTermination()
    
    print(f"\n✅ Query completed successfully!")
    print(f"   Final status: {query.status}")
else:
    print("❌ No query was started - check if shipment bronze zone is enabled")

✅ Started streaming query: bronze_shipment_stream
   Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}

✅ Query completed successfully!
   Final status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}


# Process Silver Table

In [ ]:
from arcflow.pipelines.zone_pipeline import ZonePipeline

# Create a streaming pipeline for bronze zone
pipeline = ZonePipeline(
    spark=spark,
    zone="silver",
    config={
        'streaming_enabled': True,  # Enable streaming mode
        'checkpoint_uri': r'C:\Users\milescole\source\ArcFlow\dev\storage\checkpoints'
    }
)

query = pipeline.process_table(tables["shipment"])


if query:
    print(f"✅ Started streaming query: {query.name}")
    print(f"   Status: {query.status}")
    
    # Wait for the query to finish (availableNow will complete automatically)
    query.awaitTermination()
    
    print(f"\n✅ Query completed successfully!")
    print(f"   Final status: {query.status}")
else:
    print("❌ No query was started - check if shipment silver zone is enabled")

✅ Started streaming query: silver_shipment_stream
   Query Name: silver_shipment_stream
   Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}

✅ Query completed successfully!
   Final status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}

✅ Query completed successfully!
   Final status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}


### Verify the Results

Read the bronze zone table to see the processed data:

In [61]:
df = spark.sql('select * from silver.shipment')
df.show()

+--------------------+----------------+-----------+--------------+-----------------------+-------------------+--------------------+--------------------+------------------+--------------------+----------------+-------------------+-----------------------+--------------------+---------------------+-----------------+--------------------+------------------+--------------------+------+----------+------------+-----------------------------+------+--------------------+---------------+---------------+--------------------+-------------+--------------+------------------+---------------+----------------+------------+---------------+----------------------+-------------+--------------------+--------------------+--------------------+--------------------+------+------------------+-----+---------------------+
|       enqueued_time|        producer|record_type|schema_version|committed_delivery_date|current_facility_id|         customer_id|       customer_name|    declared_value| destination_address|desti

In [91]:
spark.sql('drop schema bronze cascade')
spark.sql('drop schema silver cascade')


DataFrame[]

In [23]:
df = spark.sql('select * from silver.order')
df.show()

+--------------------+--------------------+-------------------+-----------+------+--------------------+---------------+--------------------+--------------------+--------------+--------------------+-----------+------------------+--------+--------------------+----------+-----------------+---------------------+
|            order_id|        order_number|         order_date|order_total|source|         customer_id|organization_id|        generated_at|         description|extended_price|             item_id|line_number|        net_weight|quantity|                 sku|unit_price|warranty_included|_processing_timestamp|
+--------------------+--------------------+-------------------+-----------+------+--------------------+---------------+--------------------+--------------------+--------------+--------------------+-----------+------------------+--------+--------------------+----------+-----------------+---------------------+
|eb7c44a2-f6d1-42e...|SO127482402507682...|2025-11-01 00:00:00|    880